# Lawgic Classifier Fine-Tuning Notebook

This notebook fine-tunes a transformer encoder as the local Lawgic clause classifier. It uses ToS;DR point annotations from `generated_files/tos_dr/tos_dr_points.csv`, treats `point_quote_text` as the clause text, and uses `cases.case_topic` as the ground-truth topic.

The model is configured as a **multi-label** sequence classifier even though the current source file has one topic per point. That keeps the training target compatible with future ToS;DR rows where one clause may route to more than one topic. The classifier head uses independent sigmoid outputs through Hugging Face's `problem_type="multi_label_classification"`; no softmax is used.

Outputs created by this notebook:

- Taxonomy metadata: `notebooks/model_finetuning/tosdr_classifier_taxonomy.json`
- Final model directory: `saved_models/lawgic_classifier/`
- Trainer checkpoints: `saved_models/lawgic_classifier/checkpoints/`
- Test metrics: `saved_models/lawgic_classifier/test_metrics.json`

## 1. Environment Imports and Experiment Configuration

This cell centralizes imports, file paths, and experiment constants. Keep these values near the top so changing the base encoder, split mode, sequence length, or output directory does not require editing later training logic.

The notebook discovers the project root by walking upward until it finds `generated_files/tos_dr/tos_dr_points.csv`. That makes it safe to run from either the workspace root or from the notebook directory.

In [7]:
%pip freeze > requirements.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [1]:
import inspect
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
# MODEL_NAME = "microsoft/deberta-v3-base"  # Alternative encoder to try on CUDA hardware.

MAX_LENGTH = 256
DECISION_THRESHOLD = 0.50
TRAIN_SIZE = 0.80
VAL_SIZE = 0.10
TEST_SIZE = 0.10
SPLIT_STRATEGY = "stratified"  # Options: "stratified" or "group".

LEARNING_RATE = 3e-5
MAX_EPOCHS = 20
BATCH_SIZE = 8
EARLY_STOPPING_PATIENCE = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "generated_files/tos_dr/tos_dr_points.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing generated_files/tos_dr/tos_dr_points.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "generated_files/tos_dr/tos_dr_points.csv"
TOPIC_METADATA_PATH = PROJECT_ROOT / "generated_files/tos_dr/tosdr_topics.json"
TAXONOMY_PATH = PROJECT_ROOT / "notebooks/model_finetuning/tosdr_classifier_taxonomy.json"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "saved_models/lawgic_classifier"
CHECKPOINT_DIR = MODEL_OUTPUT_DIR / "checkpoints"

random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Training data: {DATA_PATH}")
print(f"Taxonomy output: {TAXONOMY_PATH}")
print(f"Model output: {MODEL_OUTPUT_DIR}")

Project root: /Users/riki/Coding Projects/Thesis/lawgic
Training data: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/tos_dr/tos_dr_points.csv
Taxonomy output: /Users/riki/Coding Projects/Thesis/lawgic/notebooks/model_finetuning/tosdr_classifier_taxonomy.json
Model output: /Users/riki/Coding Projects/Thesis/lawgic/saved_models/lawgic_classifier


## 2. Hardware Device Autodetection and Precision Policy

The same notebook should run on both target machines:

- **ASUS Zephyrus G14 / NVIDIA CUDA**: use CUDA and enable `fp16=True` so tensor cores can accelerate transformer training.
- **M1 MacBook Pro / Apple Silicon MPS**: use MPS when available, but keep full precision. MPS mixed precision can be less stable for transformer fine-tuning and may trigger backend-specific matrix errors.
- **CPU fallback**: run in full precision for correctness, though training will be slow.

Hugging Face `Trainer` handles device placement internally. The values below are passed into `TrainingArguments` later.

In [2]:
def detect_device() -> tuple[str, torch.device]:
    if torch.cuda.is_available():
        return "cuda", torch.device("cuda")
    if torch.backends.mps.is_available():
        return "mps", torch.device("mps")
    return "cpu", torch.device("cpu")


DEVICE_TYPE, DEVICE = detect_device()
USE_FP16 = DEVICE_TYPE == "cuda"
USE_BF16 = False  # Keep MPS/CPU stable; CUDA fp16 is the intended mixed-precision path.

if DEVICE_TYPE == "cuda":
    torch.set_float32_matmul_precision("high")
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif DEVICE_TYPE == "mps":
    DEVICE_NAME = "Apple Silicon MPS"
else:
    DEVICE_NAME = "CPU"

print(f"Selected device: {DEVICE_TYPE} ({DEVICE_NAME})")
print(f"Training precision: fp16={USE_FP16}, bf16={USE_BF16}")

Selected device: mps (Apple Silicon MPS)
Training precision: fp16=False, bf16=False


## 3. Data Loading and Multi-Label Target Matrix Extraction

This notebook intentionally ignores the older `cleaned_tos_comments.csv` assumption. The supervised signal comes from the ToS;DR points export:

- `point_quote_text`: clause text to classify.
- `cases.case_topic`: Lawgic routing/classification topic.

Rows with missing text or missing topic are removed. The current file has one topic per `point_id`, but the target builder groups rows by point and creates a multi-hot vector. If a future export repeats a point with several topics, the same code will set multiple positive labels for that sample.

The saved taxonomy is built from the actual observed labels and enriched with metadata from `tosdr_topics.json` when titles match.

In [3]:
raw_df = pd.read_csv(DATA_PATH)

REQUIRED_COLUMNS = [
    "point_id",
    "point_quote_text",
    "point_document_id",
    "service_name",
    "cases.case_topic",
]
missing_columns = [column for column in REQUIRED_COLUMNS if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in {DATA_PATH}: {missing_columns}")

clean_df = raw_df.copy()
clean_df["point_quote_text"] = clean_df["point_quote_text"].astype("string").str.strip()
clean_df["cases.case_topic"] = clean_df["cases.case_topic"].astype("string").str.strip()
clean_df = clean_df.dropna(subset=["point_quote_text", "cases.case_topic"])
clean_df = clean_df[(clean_df["point_quote_text"] != "") & (clean_df["cases.case_topic"] != "")]

samples_df = (
    clean_df.groupby(["point_id", "point_quote_text"], dropna=False)
    .agg(
        point_document_id=("point_document_id", "first"),
        service_name=("service_name", "first"),
        topics=("cases.case_topic", lambda values: sorted(set(values.dropna().astype(str)))),
        source_rows=("cases.case_topic", "size"),
    )
    .reset_index()
)

samples_df["primary_topic"] = samples_df["topics"].map(lambda topics: topics[0])
samples_df["label_count"] = samples_df["topics"].map(len)

target_labels = sorted({topic for topics in samples_df["topics"] for topic in topics})
label2id = {label: idx for idx, label in enumerate(target_labels)}
id2label = {idx: label for label, idx in label2id.items()}
label_columns = [f"label_{idx:02d}_{label}" for idx, label in id2label.items()]


def encode_topics(topics: list[str]) -> np.ndarray:
    vector = np.zeros(len(target_labels), dtype=np.float32)
    for topic in topics:
        vector[label2id[topic]] = 1.0
    return vector


label_matrix = np.vstack(samples_df["topics"].map(encode_topics).to_numpy())
samples_df["label_vector"] = list(label_matrix)
for idx, column in enumerate(label_columns):
    samples_df[column] = label_matrix[:, idx]

if TOPIC_METADATA_PATH.exists():
    with TOPIC_METADATA_PATH.open("r", encoding="utf-8") as file:
        topic_metadata = json.load(file)
else:
    topic_metadata = []
metadata_by_title = {item.get("title"): item for item in topic_metadata if item.get("title")}
positive_counts = label_matrix.sum(axis=0).astype(int)

taxonomy = []
for idx, title in id2label.items():
    metadata = metadata_by_title.get(title, {})
    taxonomy.append(
        {
            "classifier_id": idx,
            "title": title,
            "tosdr_topic_id": metadata.get("id"),
            "subtitle": metadata.get("subtitle"),
            "description": metadata.get("description"),
            "positive_count": int(positive_counts[idx]),
        }
    )

taxonomy_payload = {
    "name": "lawgic_classifier_tosdr_topics",
    "source_data": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "source_topic_metadata": str(TOPIC_METADATA_PATH.relative_to(PROJECT_ROOT)) if TOPIC_METADATA_PATH.exists() else None,
    "model_name": MODEL_NAME,
    "num_labels": len(target_labels),
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "labels": taxonomy,
    "label2id": label2id,
    "id2label": {str(idx): label for idx, label in id2label.items()},
}

TAXONOMY_PATH.parent.mkdir(parents=True, exist_ok=True)
with TAXONOMY_PATH.open("w", encoding="utf-8") as file:
    json.dump(taxonomy_payload, file, indent=2, ensure_ascii=False)

display(samples_df[["point_id", "service_name", "point_quote_text", "topics", "label_count"]].head(3))
print(f"Raw rows: {len(raw_df):,}")
print(f"Usable source rows: {len(clean_df):,}")
print(f"Training samples after point grouping: {len(samples_df):,}")
print(f"Observed labels: {len(target_labels)}")
print(f"Multi-label samples: {(samples_df['label_count'] > 1).sum():,}")
print(f"Saved taxonomy to: {TAXONOMY_PATH}")

label_distribution = pd.DataFrame(
    {
        "classifier_id": range(len(target_labels)),
        "topic": target_labels,
        "positive_count": positive_counts,
    }
).sort_values("positive_count", ascending=False)
display(label_distribution)

,point_id,service_name,point_quote_text,topics,label_count
0,560,CouchSurfing,We may automatically collect information using...,[Trackers],1
1,561,CouchSurfing,"By registering for an account, you further agr...",[Anonymity],1
2,562,LastPass,We may suspend the Services or terminate the A...,[Suspension and Censorship],1


Raw rows: 23,868
Usable source rows: 22,404
Training samples after point grouping: 22,404
Observed labels: 25
Multi-label samples: 0
Saved taxonomy to: /Users/riki/Coding Projects/Thesis/lawgic/notebooks/model_finetuning/tosdr_classifier_taxonomy.json


,classifier_id,topic,positive_count
20,20,Trackers,2524
7,7,Governance,2476
8,8,Guarantee,1603
15,15,Personal Data,1575
21,21,Transparency,1381
4,4,Content,1231
19,19,Third Parties,1154
18,18,Suspension and Censorship,1149
12,12,Notice of Changing Terms,979
3,3,Changes,910


## 4. Train, Validation, and Test Split Generation

The default split is stratified by `primary_topic`. This is appropriate for the current export because each point has exactly one topic and the label distribution is imbalanced.

A `group` split is also available for stricter document-aware evaluation. It keeps all rows from the same document/service group on one side of a split, reducing leakage from near-duplicate clauses inside the same service policy. Use it when document-level generalization matters more than per-topic stratification.

For the thesis classifier training run, start with `stratified`, inspect metrics, then rerun with `group` as a robustness check.

In [4]:
def stratify_or_none(labels: pd.Series) -> pd.Series | None:
    counts = labels.value_counts()
    if len(counts) < 2 or counts.min() < 2:
        return None
    return labels


def make_stratified_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_or_none(df["primary_topic"]),
    )
    relative_val_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_or_none(train_val_df["primary_topic"]),
    )
    return train_df, val_df, test_df


def make_group_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    group_values = (
        df["point_document_id"].fillna("missing_document").astype(str)
        + "::"
        + df["service_name"].fillna("missing_service").astype(str)
    )
    first_split = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    train_val_idx, test_idx = next(first_split.split(df, groups=group_values))
    train_val_df = df.iloc[train_val_idx].copy()
    test_df = df.iloc[test_idx].copy()

    train_val_groups = group_values.iloc[train_val_idx]
    relative_val_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
    second_split = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=SEED)
    train_idx, val_idx = next(second_split.split(train_val_df, groups=train_val_groups))
    train_df = train_val_df.iloc[train_idx].copy()
    val_df = train_val_df.iloc[val_idx].copy()
    return train_df, val_df, test_df


if SPLIT_STRATEGY == "stratified":
    train_df, val_df, test_df = make_stratified_split(samples_df)
elif SPLIT_STRATEGY == "group":
    train_df, val_df, test_df = make_group_split(samples_df)
else:
    raise ValueError(f"Unknown SPLIT_STRATEGY: {SPLIT_STRATEGY}")

for split_df in (train_df, val_df, test_df):
    split_df.reset_index(drop=True, inplace=True)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "unique_primary_topics": [
            train_df["primary_topic"].nunique(),
            val_df["primary_topic"].nunique(),
            test_df["primary_topic"].nunique(),
        ],
    }
)
display(split_summary)

print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Train/validation/test rows: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

,split,rows,unique_primary_topics
0,train,17922,25
1,validation,2241,25
2,test,2241,25


Split strategy: stratified
Train/validation/test rows: 17,922 / 2,241 / 2,241


## 5. Tokenization and PyTorch Dataset Wrappers

Transformer encoders consume token IDs, attention masks, and floating-point target vectors. The dataset wrapper below keeps tokenization deterministic and uses `DataCollatorWithPadding` so each batch pads only to the longest sequence in that batch.

Labels are returned as `float32` tensors because Hugging Face uses binary cross-entropy with logits for `problem_type="multi_label_classification"`. That loss expects independent binary targets per topic.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


class LawgicClauseDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer: AutoTokenizer, max_length: int):
        self.texts = frame["point_quote_text"].astype(str).tolist()
        self.labels = np.vstack(frame["label_vector"].to_numpy()).astype(np.float32)
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            max_length=max_length,
            padding=False,
        )

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        item = {key: torch.tensor(values[index]) for key, values in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.float32)
        return item


train_dataset = LawgicClauseDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = LawgicClauseDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = LawgicClauseDataset(test_df, tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample_item = train_dataset[0]
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Train samples: {len(train_dataset):,}")
print(f"Sample tensor keys: {list(sample_item.keys())}")
print(f"Sample label shape: {tuple(sample_item['labels'].shape)}")

Tokenizer: BertTokenizerFast
Train samples: 17,922
Sample tensor keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Sample label shape: (25,)


## 6. Model Initialization

The classifier is a transformer encoder plus a linear classification head. `num_labels` is derived from the observed taxonomy, and `id2label`/`label2id` are stored inside the model config so predictions remain interpretable after saving and loading.

The important setting is `problem_type="multi_label_classification"`. It tells Hugging Face to train with independent binary targets and sigmoid-compatible logits, not mutually exclusive softmax probabilities.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(target_labels),
    problem_type="multi_label_classification",
    id2label={idx: label for idx, label in id2label.items()},
    label2id={label: idx for label, idx in label2id.items()},
)

print(f"Loaded model: {MODEL_NAME}")
print(f"Classifier labels: {model.config.num_labels}")
print(f"Problem type: {model.config.problem_type}")

## 7. Metric Computation

For multi-label classification, logits are converted with a sigmoid independently for each topic. A clause can therefore activate zero, one, or many topics. The default threshold is `0.50`, which is a standard starting point; after the first training run, threshold calibration can be explored on the validation set if recall or precision needs adjustment.

The main selection metric is **macro F1** because it gives small topics more influence than micro F1. This matters for legal taxonomy routing, where rare but important topics like business transfers or user involvement should not disappear behind high-volume privacy/tracking categories.

In [ ]:
def sigmoid(logits: np.ndarray) -> np.ndarray:
    clipped_logits = np.clip(logits, -60, 60)
    return 1.0 / (1.0 + np.exp(-clipped_logits))


def logits_to_predictions(logits: np.ndarray, threshold: float = DECISION_THRESHOLD) -> np.ndarray:
    return (sigmoid(logits) >= threshold).astype(int)


def compute_metrics(eval_pred) -> dict[str, float]:
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    labels = labels.astype(int)
    predictions = logits_to_predictions(logits)

    return {
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, predictions, average="micro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "subset_accuracy": accuracy_score(labels, predictions),
        "predicted_positive_rate": float(predictions.mean()),
    }


def make_classification_report(logits: np.ndarray, labels: np.ndarray) -> dict:
    predictions = logits_to_predictions(logits)
    return classification_report(
        labels.astype(int),
        predictions,
        target_names=target_labels,
        output_dict=True,
        zero_division=0,
    )

print(f"Decision threshold: {DECISION_THRESHOLD}")
print(f"Primary checkpoint metric: eval_macro_f1")

## 8. Training Configuration and Execution

The training defaults follow the requested scientific grounding for unfair/legal clause assessment:

- AdamW optimizer path via `optim="adamw_torch"`.
- Initial learning rate: `3e-5`.
- Stable per-device batch size: `8`.
- Upper epoch limit: `20`.
- Early stopping patience: `3` validation evaluations.
- Best model selected by `eval_macro_f1`.

The helper below handles small naming differences across `transformers` versions (`eval_strategy` vs `evaluation_strategy`) so the notebook is more likely to run on both machines without manual edits.

In [ ]:
def build_training_arguments() -> TrainingArguments:
    signature = inspect.signature(TrainingArguments.__init__)
    parameter_names = set(signature.parameters)

    kwargs = {
        "output_dir": str(CHECKPOINT_DIR),
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "num_train_epochs": MAX_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_macro_f1",
        "greater_is_better": True,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": USE_FP16,
        "bf16": USE_BF16,
        "optim": "adamw_torch",
    }

    if "eval_strategy" in parameter_names:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    supported_kwargs = {key: value for key, value in kwargs.items() if key in parameter_names}
    return TrainingArguments(**supported_kwargs)


training_args = build_training_arguments()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(training_args)
train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

## 9. Test Evaluation and Final Weight Storage

After training, the `Trainer` has already restored the checkpoint with the best validation macro F1. This final cell evaluates that best model on the held-out test split, writes machine-readable metrics, and saves all reloadable artifacts.

Saved artifacts include:

- Fine-tuned encoder/classifier weights.
- Tokenizer files.
- Model config with `id2label`, `label2id`, and `problem_type`.
- Taxonomy JSON.
- Test metrics and per-topic classification report.

These files are enough for a later Lawgic inference notebook or application module to load the classifier and route clauses by topic probabilities.

In [ ]:
def to_jsonable(value):
    if isinstance(value, dict):
        return {str(key): to_jsonable(inner_value) for key, inner_value in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value


MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
test_predictions = trainer.predict(test_dataset)
test_logits = test_predictions.predictions[0] if isinstance(test_predictions.predictions, tuple) else test_predictions.predictions
test_labels = np.vstack(test_df["label_vector"].to_numpy()).astype(int)
test_probabilities = sigmoid(test_logits)
test_report = make_classification_report(test_logits, test_labels)

trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

artifact_payload = {
    "model_name": MODEL_NAME,
    "model_output_dir": str(MODEL_OUTPUT_DIR.relative_to(PROJECT_ROOT)),
    "taxonomy_path": str(TAXONOMY_PATH.relative_to(PROJECT_ROOT)),
    "checkpoint_dir": str(CHECKPOINT_DIR.relative_to(PROJECT_ROOT)),
    "split_strategy": SPLIT_STRATEGY,
    "decision_threshold": DECISION_THRESHOLD,
    "test_metrics": test_metrics,
    "classification_report": test_report,
    "label2id": label2id,
    "id2label": {str(idx): label for idx, label in id2label.items()},
}

metrics_path = MODEL_OUTPUT_DIR / "test_metrics.json"
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(to_jsonable(artifact_payload), file, indent=2, ensure_ascii=False)

thresholds_path = MODEL_OUTPUT_DIR / "classification_thresholds.json"
with thresholds_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "default_threshold": DECISION_THRESHOLD,
            "per_label_thresholds": {label: DECISION_THRESHOLD for label in target_labels},
        },
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved final model and tokenizer to: {MODEL_OUTPUT_DIR}")
print(f"Saved test metrics to: {metrics_path}")
print(f"Saved classification thresholds to: {thresholds_path}")
print(f"Mean predicted probability on test set: {test_probabilities.mean():.4f}")
display(pd.DataFrame([test_metrics]))